# Batch Inference: Fashion Recommendations

This notebook performs batch inference using registered models from Unity Catalog to generate recommendations for all customers.

**Features:**
- Loads all available models (popularity, age_rules, lstm, ensemble) from Unity Catalog
- Only processes models with 'Champion' alias
- Skips models without the champion alias
- Generates predictions for each champion model
- Writes results to model-specific Gold tables in Delta format
- Syncs predictions to Lakebase synced tables for OLTP access

**Inputs:**
- catalog_name: Unity Catalog name
- schema_name: Schema containing tables and models

**Outputs:**
- Model-specific Delta tables (Gold layer):
  - popularity_predictions_gold
  - age_rules_predictions_gold
  - lstm_predictions_gold
  - ensemble_predictions_gold

- Model-specific synced tables (Lakebase OLTP):
  - popularity_predictions_synced
  - age_rules_predictions_synced
  - lstm_predictions_synced
  - ensemble_predictions_synced

**Lakebase Instance:** shared-online-store

## Setup

In [ ]:
import sys

# Add project root to path (go up 3 levels from notebooks/)
sys.path.append("../../../")

import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StringType, StructType, StructField, IntegerType, DoubleType
from datetime import datetime
import mlflow
from mlflow.tracking import MlflowClient
import pandas as pd

from config.widget_utils import get_widget_or_default
from config.catalog_config import get_table_config

In [ ]:
# Get parameters from bundle (passed as notebook parameters)
# Falls back to defaults when running interactively
catalog_name = get_widget_or_default("catalog_name", "jongseob_demo")
schema_name = get_widget_or_default("schema_name", "dev_fashion_recommendations")

print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")

# Get table config
tables = get_table_config(catalog_name, schema_name)

## Define Model Configuration

In [ ]:
# Model configuration: name -> output table mapping
MODEL_CONFIG = {
    "popularity_model": {
        "output_table": f"{catalog_name}.{schema_name}.popularity_predictions_gold",
        "synced_table": f"{catalog_name}.{schema_name}.popularity_predictions_synced",
        "description": "Popularity baseline model"
    },
    "age_rules_model": {
        "output_table": f"{catalog_name}.{schema_name}.age_rules_predictions_gold",
        "synced_table": f"{catalog_name}.{schema_name}.age_rules_predictions_synced",
        "description": "Age-based rules model"
    },
    "lstm_model": {
        "output_table": f"{catalog_name}.{schema_name}.lstm_predictions_gold",
        "synced_table": f"{catalog_name}.{schema_name}.lstm_predictions_synced",
        "description": "LSTM sequential model"
    },
    "ensemble_model": {
        "output_table": f"{catalog_name}.{schema_name}.ensemble_predictions_gold",
        "synced_table": f"{catalog_name}.{schema_name}.ensemble_predictions_synced",
        "description": "Ensemble blend model"
    }
}

# Lakebase instance for synced tables (OLTP access)
LAKEBASE_INSTANCE = "shared-online-store"

print(f"Configured {len(MODEL_CONFIG)} models for batch inference")
print(f"Lakebase instance: {LAKEBASE_INSTANCE}")

## Helper Functions

In [ ]:
def check_model_alias(catalog_name: str, schema_name: str, model_name: str, alias: str = "Champion") -> tuple:
    """
    Check if a model has the specified alias in Unity Catalog.
    
    Args:
        catalog_name: Catalog name
        schema_name: Schema name
        model_name: Model name
        alias: Alias to check (default: Champion)
        
    Returns:
        Tuple of (has_alias: bool, version: str or None)
    """
    client = MlflowClient()
    full_model_name = f"{catalog_name}.{schema_name}.{model_name}"
    
    try:
        # Get model version by alias
        model_version = client.get_model_version_by_alias(full_model_name, alias)
        print(f"✓ Model '{model_name}' has '{alias}' alias (version {model_version.version})")
        return True, model_version.version
    except Exception as e:
        print(f"✗ Model '{model_name}' does not have '{alias}' alias - skipping")
        return False, None


def load_model_with_alias(catalog_name: str, schema_name: str, model_name: str, alias: str = "Champion"):
    """
    Load model from Unity Catalog using specified alias.
    
    Args:
        catalog_name: Catalog name
        schema_name: Schema name
        model_name: Model name
        alias: Alias to load (default: Champion)
        
    Returns:
        Loaded MLflow model
    """
    model_uri = f"models:/{catalog_name}.{schema_name}.{model_name}@{alias}"
    print(f"Loading model from: {model_uri}")
    model = mlflow.pyfunc.load_model(model_uri)
    return model

In [ ]:
def create_or_update_synced_table(
    source_table: str, 
    synced_table: str, 
    lakebase_instance: str,
    primary_key: list = ["customer_id"]
):
    """
    Create or update a synced table in Lakebase for OLTP access.
    
    Args:
        source_table: Source Delta table name (fully qualified)
        synced_table: Synced table name (fully qualified)
        lakebase_instance: Lakebase instance name (e.g., "shared-online-store")
        primary_key: List of columns that form the primary key
    """
    try:
        print(f"Creating synced table: {synced_table}")
        print(f"  Source: {source_table}")
        print(f"  Database instance: {lakebase_instance}")
        print(f"  Primary key: {primary_key}")
        
        from databricks.sdk import WorkspaceClient
        from databricks.sdk.service.database import (
            SyncedDatabaseTable, 
            SyncedTableSpec, 
            NewPipelineSpec, 
            SyncedTableSchedulingPolicy
        )
        
        w = WorkspaceClient()
        
        # Delete existing synced table if it exists
        try:
            print(f"  Checking for existing synced table...")
            w.database.delete_synced_database_table(name=synced_table)
            print(f"  Deleted existing synced table")
        except Exception as e:
            # Table doesn't exist, which is fine
            print(f"  No existing table to delete")
        
        # Parse catalog and schema from synced table name
        synced_parts = synced_table.split('.')
        catalog = synced_parts[0]
        schema = synced_parts[1]
        
        # Create synced table using the correct API
        print(f"  Creating synced table...")
        synced_table_obj = w.database.create_synced_database_table(
            SyncedDatabaseTable(
                name=synced_table,
                database_instance_name=lakebase_instance,  # e.g., "shared-online-store"
                logical_database_name=f"{catalog}_{schema}",  # Logical database name in the instance
                spec=SyncedTableSpec(
                    source_table_full_name=source_table,
                    primary_key_columns=primary_key,
                    scheduling_policy=SyncedTableSchedulingPolicy.TRIGGERED,  # Batch/triggered mode
                    create_database_objects_if_missing=True,  # Create database/schema if needed
                    new_pipeline_spec=NewPipelineSpec(
                        storage_catalog=catalog,
                        storage_schema=schema
                    )
                )
            )
        )
        
        print(f"✓ Synced table created successfully: {synced_table}")
        print(f"  Name: {synced_table_obj.name}")
        if hasattr(synced_table_obj, 'data_synchronization_status'):
            print(f"  Status: {synced_table_obj.data_synchronization_status.detailed_state}")
        
        return True
        
    except Exception as e:
        print(f"✗ Error creating synced table: {str(e)}")
        import traceback
        traceback.print_exc()
        return False

## Load Input Data

In [ ]:
# Load customer data for scoring
print(f"Loading customers from: {tables.CUSTOMERS_BRONZE}")
customers_df = spark.table(tables.CUSTOMERS_BRONZE)
num_customers = customers_df.count()
print(f"Loaded {num_customers} customers")

# Load articles for reference
print(f"\nLoading articles from: {tables.ARTICLES_BRONZE}")
articles_df = spark.table(tables.ARTICLES_BRONZE)
num_articles = articles_df.count()
print(f"Loaded {num_articles} articles")

# Load customer features (needed for age_rules model)
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()

print(f"\nLoading customer features from: {tables.CUSTOMER_FEATURES}")
customer_features_df = fe.read_table(name=tables.CUSTOMER_FEATURES)
num_features = customer_features_df.count()
print(f"Loaded customer features for {num_features} customers")

## Batch Inference for All Champion Models

In [ ]:
# Track inference results
inference_results = []

print("="*80)
print("BATCH INFERENCE: Checking models with Champion alias")
print("="*80)

for model_name, config in MODEL_CONFIG.items():
    print(f"\n{'='*80}")
    print(f"Processing: {model_name} ({config['description']})")
    print(f"{'='*80}")
    
    # Check if model has Champion alias and get version
    has_alias, model_version = check_model_alias(catalog_name, schema_name, model_name, alias="Champion")
    if not has_alias:
        print(f"Skipping {model_name}\n")
        inference_results.append({
            "model_name": model_name,
            "status": "SKIPPED",
            "reason": "No Champion alias",
            "num_predictions": 0
        })
        continue
    
    try:
        # Load model with Champion alias
        model = load_model_with_alias(catalog_name, schema_name, model_name, alias="Champion")
        
        # Prepare input data based on model type
        print(f"Preparing input data for {model_name}...")
        
        if model_name == "age_rules_model":
            # Age rules model needs customer_id AND age_group
            input_df = customers_df.select("customer_id").join(
                customer_features_df.select("customer_id", "age_group"),
                on="customer_id",
                how="left"
            )
            # Filter out customers without age group
            input_df = input_df.filter(F.col("age_group").isNotNull())
            print(f"  Input features: customer_id, age_group")
        else:
            # Other models only need customer_id
            input_df = customers_df.select("customer_id")
            print(f"  Input features: customer_id")
        
        # Convert to pandas for model prediction
        input_pd = input_df.toPandas()
        print(f"  Input rows: {len(input_pd):,}")
        
        # Generate predictions
        print(f"Generating predictions...")
        predictions_pd = model.predict(input_pd)
        
        # Convert predictions back to Spark DataFrame
        predictions_df = spark.createDataFrame(predictions_pd)
        
        # Add metadata columns including version and alias
        predictions_df = predictions_df \
            .withColumn("model_name", F.lit(model_name)) \
            .withColumn("model_version", F.lit(str(model_version))) \
            .withColumn("model_alias", F.lit("Champion")) \
            .withColumn("prediction_timestamp", F.current_timestamp()) \
            .withColumn("batch_id", F.lit(datetime.now().strftime("%Y%m%d_%H%M%S")))
        
        # Write predictions to Delta table
        output_table = config["output_table"]
        print(f"Writing predictions to: {output_table}")
        
        predictions_df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(output_table)
        
        num_predictions = predictions_df.count()
        print(f"✓ Successfully wrote {num_predictions} predictions to {output_table}")
        
        # Enable Change Data Feed for synced tables
        print(f"Enabling Change Data Feed on {output_table}...")
        spark.sql(f"""
            ALTER TABLE {output_table} 
            SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
        """)
        print(f"✓ Change Data Feed enabled")
        
        # Display sample predictions
        print("\nSample predictions:")
        display(predictions_df.limit(5))
        
        inference_results.append({
            "model_name": model_name,
            "model_version": model_version,
            "model_alias": "Champion",
            "status": "SUCCESS",
            "reason": "Completed",
            "num_predictions": num_predictions,
            "output_table": output_table
        })
        
    except Exception as e:
        print(f"✗ Error processing {model_name}: {str(e)}")
        import traceback
        traceback.print_exc()
        
        inference_results.append({
            "model_name": model_name,
            "model_version": model_version if 'model_version' in locals() else None,
            "model_alias": "Champion",
            "status": "FAILED",
            "reason": str(e),
            "num_predictions": 0
        })

print("\n" + "="*80)
print("BATCH INFERENCE COMPLETED")
print("="*80)

## Sync to Lakebase OLTP Tables

In [ ]:
# Sync all successfully processed models to Lakebase
sync_results = []

print("="*80)
print("SYNCING TO LAKEBASE: Creating synced tables for OLTP access")
print("="*80)

for result in inference_results:
    if result["status"] == "SUCCESS":
        model_name = result["model_name"]
        config = MODEL_CONFIG[model_name]
        output_table = config["output_table"]
        synced_table = config["synced_table"]
        
        print(f"\n{'='*80}")
        print(f"Syncing: {model_name}")
        print(f"{'='*80}")
        
        # Create or update synced table
        sync_success = create_or_update_synced_table(
            source_table=output_table,
            synced_table=synced_table,
            lakebase_instance=LAKEBASE_INSTANCE,
            primary_key=["customer_id"]
        )
        
        if sync_success:
            sync_results.append({
                "model_name": model_name,
                "output_table": output_table,
                "synced_table": synced_table,
                "status": "SYNCED"
            })
        else:
            sync_results.append({
                "model_name": model_name,
                "output_table": output_table,
                "synced_table": synced_table,
                "status": "SYNC_FAILED"
            })
    else:
        print(f"Skipping sync for {result['model_name']} (status: {result['status']})")

print("\n" + "="*80)
print("LAKEBASE SYNC COMPLETED")
print("="*80)

# Display sync summary
if sync_results:
    sync_df = pd.DataFrame(sync_results)
    print(f"\nSynced {len([r for r in sync_results if r['status'] == 'SYNCED'])} table(s) to Lakebase")
    print("\nSync Results:")
    display(sync_df)
else:
    print("\nNo tables to sync (no successful batch inference runs)")


## Summary Report

In [ ]:
# Display summary statistics
import pandas as pd

summary_df = pd.DataFrame(inference_results)
print("\n" + "="*80)
print("BATCH INFERENCE SUMMARY")
print("="*80)
print(f"Catalog: {catalog_name}")
print(f"Schema: {schema_name}")
print(f"Total Models Checked: {len(MODEL_CONFIG)}")
print(f"Models Processed: {len([r for r in inference_results if r['status'] == 'SUCCESS'])}")
print(f"Models Skipped: {len([r for r in inference_results if r['status'] == 'SKIPPED'])}")
print(f"Models Failed: {len([r for r in inference_results if r['status'] == 'FAILED'])}")
print("\nDetailed Results:")
print("="*80)

# Show key columns in summary
if 'model_version' in summary_df.columns:
    display(summary_df[['model_name', 'model_version', 'model_alias', 'status', 'num_predictions', 'reason']])
else:
    display(summary_df)

# Return success if at least one model was processed successfully
success_count = len([r for r in inference_results if r['status'] == 'SUCCESS'])
if success_count == 0:
    raise Exception("No models were processed successfully. Please check that models have Champion alias.")
else:
    print(f"\n✓ Batch inference completed successfully for {success_count} model(s)")
    print("\n" + "="*80)
    print("OUTPUT SCHEMA")
    print("="*80)
    print("\nDelta tables (Gold layer) contain:")
    print("  - customer_id: Customer identifier")
    print("  - predicted_articles: List of recommended article IDs")
    print("  - model_name: Name of the model used")
    print("  - model_version: Version number of the model")
    print("  - model_alias: Alias used (Champion)")
    print("  - prediction_timestamp: When predictions were generated")
    print("  - batch_id: Batch identifier")
    print("\nSynced tables (Lakebase OLTP) mirror the Delta tables with:")
    print("  - Primary key: customer_id")
    print("  - Instance: shared-online-store")
    print("  - Access: Available for low-latency OLTP queries")